In [1]:
import pandas as pd
from pandas import CategoricalDtype
import geopandas as gpd
import json
import os

#show all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## Some preprocessing based on your chosen metro
First, some functions we'll need later on

In [2]:
#calculate the share each occupation contributes to the total employment in the state and metro
def get_employment_rate(df,statewide=False):
    total_emp = df[df['occupation_code'] == '000000']['employment'].values[0]
    df['employment_rate'] = (df['employment'] / total_emp)*10_000
    if statewide:
        return df[['occupation_code','employment_rate']].rename(columns={'employment_rate': 'state_emp_rate'})
    return df


#slice out metro-level scores foryour main market metro and get similarly-sized metros for comparison
def get_similarly_sized_metros(metro):
    total_emp = metro_scores.loc[metro_scores['occupation_code'] == '000000']
    total_emp = total_emp.sort_values('employment', ascending=False).reset_index(drop=True)
    #get the 2 metros above and below our selected metro
    selected_metro_index = total_emp[total_emp['area_name'] == metro].index[0]
    #if selected metro is the top metro, grab 4 below
    if selected_metro_index == 0:
        similar_metros = total_emp[selected_metro_index:selected_metro_index+5]
    #if it's number 2, grab the one above and then 3 below
    elif selected_metro_index == 1:
        similar_metros = total_emp[selected_metro_index-1:selected_metro_index+4]
    #if it's the bottom one, grab 4 above
    elif selected_metro_index == len(total_emp) - 1:
        similar_metros = total_emp[selected_metro_index-4:selected_metro_index+1]
    #if it's the second to last, grab 3 above and one below
    elif selected_metro_index == len(total_emp) - 2:
        similar_metros = total_emp[selected_metro_index-3:selected_metro_index+2]
    #otherwise grab two above and two below
    else:
        similar_metros = total_emp.iloc[max(0, selected_metro_index-2):selected_metro_index+3]
    
    #remove the actual key metro from the list of similars
    similar_metros = similar_metros[similar_metros['area_name'] != metro]
    
    return similar_metros['area_name'].tolist()

def print_unique_occupations(your_main_metro, your_similar_metros_list, show_similar=False, filters=None, sort_col='employment', list_size=5):

    unique_to_here = []
    found_in_similar = []

    if filters == 'high ai, high employment':
        your_main_metro = your_main_metro[(your_main_metro['ai_exposure_category'] == 'High')&(your_main_metro['employment_category'].isin(['High','Medium high']))]
    elif filters == 'highish ai, high employment':
        your_main_metro = your_main_metro[(your_main_metro['ai_exposure_category'].isin(['High','Medium high']))&(your_main_metro['employment_category'].isin(['High','Medium high']))]
        
    occs = your_main_metro.sort_values(sort_col, ascending=False)[med_cols].head(list_size)['occupation_name']
    main_metro_occs = your_main_metro['occupation_name'].values
    
    similar_metro_occs = []
    similar_metro_dfs_list = []
    for metro_area in your_similar_metros_list:
            if filters is None:
                your_similar_metros = metro_scores[metro_scores['area_name'] == metro_area]
            elif filters == 'high ai, high employment':
                your_similar_metros = metro_scores[metro_scores['area_name'] == metro_area]
                your_similar_metros = your_similar_metros[(your_similar_metros['ai_exposure_category'] == 'High')&(your_similar_metros['employment_category'].isin(['High','Medium high']))]
            elif filters == 'highish ai, high employment':
                your_similar_metros = metro_scores[metro_scores['area_name'] == metro_area]
                your_similar_metros = your_similar_metros[(your_similar_metros['ai_exposure_category'].isin(['High','Medium high']))&(your_similar_metros['employment_category'].isin(['High','Medium high']))]
            this_metro_top = your_similar_metros.sort_values(sort_col, ascending=False)[min_cols].head(list_size)
            if show_similar:
                similar_metro_dfs_list.append(this_metro_top)
            this_metro_top_occs = this_metro_top['occupation_name'].values
            similar_metro_occs.extend(this_metro_top_occs)
    similar_metro_dfs = pd.concat(similar_metro_dfs_list)
    
    print('There are ' + str(len(your_main_metro)) + ' occupations in the main metro after applying filters. Here are the top ' + str(list_size) + ' occupations based on ' + sort_col + ':')
    for occ in occs:
        if occ in set(similar_metro_occs):
            found_in_similar.append(occ)
        else:
            unique_to_here.append(occ)
        
        jobs = your_main_metro[your_main_metro['occupation_name'] == occ]['employment'].values[0]
        jobs_formatted = "{:,}".format(int(jobs)) if jobs > 0 else '0'
        job_rate = your_main_metro[your_main_metro['occupation_name'] == occ]['employment_rate'].values[0]
        #format thousands and one decimal place
        job_rate_formatted = "{:,.1f}".format(float(job_rate))
        state_rate = your_main_metro[your_main_metro['occupation_name'] == occ]['state_emp_rate'].values[0]
        state_rate_formatted = "{:,.1f}".format(float(state_rate))
        state_compare = '🟢⬆️' if job_rate > state_rate else '🔴⬇️'
        ai_rating = your_main_metro[your_main_metro['occupation_name'] == occ]['study_rating'].values[0]
        ai_rating_formatted = "{:,.2f}".format(float(ai_rating))
        #format for thousands and convert to int for easier reading
        print('')
        print(f"{occ}")
        print(f"{jobs_formatted} workers")
        print(f"{job_rate_formatted} jobs per 10k workers vs. statewide rate of {state_rate_formatted} ({state_compare})")
        print(f"{ai_rating_formatted} AI exposure rating")
                
    if len(unique_to_here) > 0:
        print(f"Occupations in top {list_size} based on {sort_col} of main metro that aren't in top {list_size} of any similar metros: {set(unique_to_here)}")
    else:
        print(f"All occupations in the top {list_size} based on {sort_col} of the main metro are also in the top {list_size} of at least one similar metro.")
        
    if len(similar_metro_dfs) > 0:
        print(f"Top {list_size} occupations based on {sort_col} for similar metros:")
        display(similar_metro_dfs)

And now import our data and slice up into dfs for each type of geography

In [3]:
#import our joined data
dtypes = {'series_id':str, 'year':int, 'period':str, 'areatype_code':str, 'state_code':str,
       'area_code':str, 'area_name':str, 'occupation_code':str, 'occupation_name':str,
       'employment':float, 'study_rating':float, 'footnote_codes':str, 'is_all_occupations':bool,
       'has_released_employment':bool, 'NEM Code':str, 'nem_merge':str,
       'ai_exposure_category':str, 'employment_category':str}
oews_with_scores = pd.read_csv('../data/processed/bls_occ_employment_w_study_scores_human_rating_beta.csv', dtype=dtypes)

#make sure the ai_exposure_category and employment_category use the size_order category type
size_order = CategoricalDtype(categories=['Low', 'Medium low', 'Medium high', 'High'], ordered=True)
oews_with_scores['ai_exposure_category'] = oews_with_scores['ai_exposure_category'].astype(size_order)
oews_with_scores['employment_category'] = oews_with_scores['employment_category'].astype(size_order)

#split into geo data types
metro_scores = oews_with_scores.loc[oews_with_scores['areatype_code'] == 'M']
state_scores = oews_with_scores.loc[oews_with_scores['areatype_code'] == 'S']
national_scores = oews_with_scores.loc[oews_with_scores['areatype_code'] == 'N']

In [5]:
#use this to find exact spelling of your market metro name
metro_scores.loc[metro_scores['area_name'].str.contains('Houston')]['area_name'].unique()

<StringArray>
['Houston-Pasadena-The Woodlands, TX']
Length: 1, dtype: str

Here's where we establish which market we're interested in looking at right now. This code block is down here instaed of up higher so you can identify the exact naming of your metro in the data.

In [6]:
##########
# CHANGE FOR YOUR MARKET
##########
market_state = 'Texas'
market_state_abbr = 'TX'
market_metro = 'Houston-Pasadena-The Woodlands, TX'
market_metro_abbr = 'hc'
custom_metro_list = []

A last bit o' processing before we start answering questions. Basically creating different dfs that will hold:
- data for metros in our market state
- data for our main metro
- a list of similary sized metros from around the country, based on total employment counts
- data for our state as a whole
- additional information about share of jobs in each occupation 

In [7]:
#make an output folder for our state if it doesn't already exist
os.makedirs(f'../data/output/{market_state_abbr.lower()}', exist_ok=True)

#slice out metro-level scores for metros that are in your market state
market_metro_scores = metro_scores[metro_scores['area_name'].str.contains(', '+market_state_abbr)]
if len(custom_metro_list)>0:
    print('found custom metros')
    custom_metro_scores = metro_scores[metro_scores['area_name'].isin(custom_metro_list)]
    market_metro_scores = pd.concat([market_metro_scores, custom_metro_scores]).drop_duplicates()
market_metro_scores = market_metro_scores.sort_values('employment', ascending=False)
market_metro_list = market_metro_scores['area_name'].unique().tolist()

#also going to create a list of the top metros in Texas to see if anything comes of those comparisons
top_texas_metros = ['Dallas-Fort Worth-Arlington, TX',
                    'Austin-Round Rock-San Marcos, TX',
                    'San Antonio-New Braunfels, TX',
                    'El Paso, TX']

#slice out the metro-level scores for your main market metro and get similarly-sized metros for comparison
main_metro_scores = market_metro_scores[market_metro_scores['area_name'] == market_metro]
similar_metros = get_similarly_sized_metros(market_metro)

#slice out the state-level scores for your market state
market_state_scores = state_scores[state_scores['area_name'] == market_state]

#get employment shares for each occupation in the state and metro
market_state_shares = get_employment_rate(market_state_scores,statewide=True)
metro_markets_dfs = []
print('TAKE A LOOK AT THESE TO MAKE SURE THERE AREN\'T ANY WEIRD ONES IN THERE')
for metro in market_metro_list:
    #printing so we can spot anything weird since our matches are kinda squishy
    print(metro)
    metro_join = get_employment_rate(metro_scores[metro_scores['area_name'] == metro])
    metro_state_join = metro_join.merge(market_state_shares, on='occupation_code', how='left')
    metro_state_join['state_metro_diff'] = metro_state_join['employment_rate'] - metro_state_join['state_emp_rate']
    metro_markets_dfs.append(metro_state_join)
market_metro_scores = pd.concat(metro_markets_dfs)

#and just redefine the main metro scores with the employment shares and differences for easier use later
main_metro_scores = market_metro_scores[market_metro_scores['area_name'] == market_metro]

TAKE A LOOK AT THESE TO MAKE SURE THERE AREN'T ANY WEIRD ONES IN THERE
Dallas-Fort Worth-Arlington, TX
Houston-Pasadena-The Woodlands, TX
Austin-Round Rock-San Marcos, TX
San Antonio-New Braunfels, TX
El Paso, TX
McAllen-Edinburg-Mission, TX
Corpus Christi, TX
Lubbock, TX
Beaumont-Port Arthur, TX
Brownsville-Harlingen, TX
Killeen-Temple, TX
College Station-Bryan, TX
Waco, TX
Amarillo, TX
Midland, TX
Longview, TX
Tyler, TX
Laredo, TX
Odessa, TX
Abilene, TX
Texarkana, TX-AR
Wichita Falls, TX
Sherman-Denison, TX
San Angelo, TX
Victoria, TX
Eagle Pass, TX


In [8]:
main_metro_scores.loc[main_metro_scores['occupation_code'].str.contains('3111', case=False)]

,series_id,year,period,areatype_code,state_code,area_code,area_name,occupation_code,occupation_name,employment,study_rating,footnote_codes,is_all_occupations,has_released_employment,NEM Code,nem_merge,soc_code_study,ai_exposure_category,employment_category,top_occ_code,top_occ_name,employment_rate,state_emp_rate,state_metro_diff
16,OEUM002642000000011311101,2025,A01,M,48,0026420,"Houston-Pasadena-The Woodlands, TX",113111,Compensation and Benefits Managers,480.0,0.460526,NaN,False,True,11-3111,113111,11-3111.00,Medium low,Medium low,110000,Management Occupations,1.459091,1.584802,-0.125711
45,OEUM002642000000013111101,2025,A01,M,48,0026420,"Houston-Pasadena-The Woodlands, TX",131111,Management Analysts,9050.0,0.500000,NaN,False,True,13-1111,131111,13-1111.00,Medium low,High,130000,Business and Financial Operations Occupations,27.509940,36.237240,-8.727300
336,OEUM002642000000031112001,2025,A01,M,48,0026420,"Houston-Pasadena-The Woodlands, TX",311120,Home Health and Personal Care Aides,63820.0,0.038462,NaN,False,True,31-1120,311120,31-1121.00,Low,High,310000,Healthcare Support Occupations,193.998273,234.230867,-40.232593
337,OEUM002642000000031113101,2025,A01,M,48,0026420,"Houston-Pasadena-The Woodlands, TX",311131,Nursing Assistants,19230.0,0.135593,NaN,False,True,31-1131,311131,31-1131.00,Low,High,310000,Healthcare Support Occupations,58.454823,63.022523,-4.567700
338,OEUM002642000000031113201,2025,A01,M,48,0026420,"Houston-Pasadena-The Woodlands, TX",311132,Orderlies,750.0,0.046875,NaN,False,True,31-1132,311132,31-1132.00,Low,Medium low,310000,Healthcare Support Occupations,2.279829,1.648762,0.631067


## Questions to answer:
- Which metros are similar to mine?
- Which occupations in my market have the highest employment numbers?
- Which occupations in my market have the highest AI exposure score?
- Which occupations have the worst combination of high AI exposure and large employment?
- What are the occupations in this metro that have a higher share of jobs than the state/nation?
- What share of total jobs in the metro have high exposure? Medium? Low?
- How does my metro's share of AI-exposed jobs compare to similarly-sized metros?
- Which top-level occupations (health care, service industry, etc) are most exposed in my metro? Statewide? Nationwide?

### What is "high" for AI exposure? Top is 1 (100% of tasks) so let's segment in 4 pieces:
- 0 - .25 = low
- .26 - .5 = medium low
- .51 - .75 = medium high
- .76 - 1 = high

### And what is "large" for exployment? 
Should this be relative to all of the occupations? Like quartiles or whatever? Probably. Let's see how that looks though. We might wanna slice it a different way.
- 1st quartile = low
- 2nd quartile = medium low
- 3rd quartile = medium high
- 4th quartile = high

In [9]:
#adjust as you need/want to show more/fewer important columns in your summary anaylsis below
min_cols = ['area_name','occupation_name','employment','study_rating', 
            'ai_exposure_category','employment_category']
med_cols = min_cols + ['employment_rate','state_emp_rate','state_metro_diff','top_occ_name']

## Which metros are similar to mine?

In [10]:
similar_metro_total_emp = metro_scores[((metro_scores['area_name'].isin(similar_metros))|(metro_scores['area_name'] == market_metro))&(metro_scores['occupation_code'] == '000000')]
similar_metro_total_emp.sort_values('employment', ascending=False)[['area_name','employment']]

,area_name,employment
31085,"Chicago-Naperville-Elgin, IL-IN",4513280.0
39247,"Dallas-Fort Worth-Arlington, TX",4049800.0
71275,"Houston-Pasadena-The Woodlands, TX",3289720.0
182647,"Washington-Arlington-Alexandria, DC-VA-MD-WV",3136190.0
129544,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",2897830.0


## Which occupations in my market have the highest employment numbers?

In [11]:
main_metro_scores.sort_values('employment', ascending=False).head(6)

,series_id,year,period,areatype_code,state_code,area_code,area_name,occupation_code,occupation_name,employment,study_rating,footnote_codes,is_all_occupations,has_released_employment,NEM Code,nem_merge,soc_code_study,ai_exposure_category,employment_category,top_occ_code,top_occ_name,employment_rate,state_emp_rate,state_metro_diff
0,OEUM002642000000000000001,2025,A01,M,48,0026420,"Houston-Pasadena-The Woodlands, TX",000000,All Occupations,3289720.0,NaN,NaN,True,True,NaN,NaN,NaN,NaN,High,0,All Occupations,10000.000000,10000.000000,0.000000
441,OEUM002642000000043000001,2025,A01,M,48,0026420,"Houston-Pasadena-The Woodlands, TX",430000,Office and Administrative Support Occupations,379010.0,NaN,NaN,False,True,NaN,NaN,NaN,NaN,High,430000,Office and Administrative Support Occupations,1152.104130,1226.537116,-74.432985
667,OEUM002642000000053000001,2025,A01,M,48,0026420,"Houston-Pasadena-The Woodlands, TX",530000,Transportation and Material Moving Occupations,314010.0,NaN,NaN,False,True,NaN,NaN,NaN,NaN,High,530000,Transportation and Material Moving Occupations,954.518926,919.334298,35.184628
371,OEUM002642000000035000001,2025,A01,M,48,0026420,"Houston-Pasadena-The Woodlands, TX",350000,Food Preparation and Serving Related Occupations,307980.0,NaN,NaN,False,True,NaN,NaN,NaN,NaN,High,350000,Food Preparation and Serving Related Occupations,936.189098,923.328283,12.860815
1,OEUM002642000000011000001,2025,A01,M,48,0026420,"Houston-Pasadena-The Woodlands, TX",110000,Management Occupations,277940.0,NaN,NaN,False,True,NaN,NaN,NaN,NaN,High,110000,Management Occupations,844.874336,862.594129,-17.719793
422,OEUM002642000000041000001,2025,A01,M,48,0026420,"Houston-Pasadena-The Woodlands, TX",410000,Sales and Related Occupations,270110.0,NaN,NaN,False,True,NaN,NaN,NaN,NaN,High,410000,Sales and Related Occupations,821.072918,852.964503,-31.891585


In [12]:

emp_cols = ['area_name','occupation_name','employment',
            'employment_rate','state_emp_rate','study_rating']

for occ in main_metro_scores.loc[~main_metro_scores['occupation_code'].str.endswith('0000')].sort_values('employment', ascending=False)[emp_cols].head(6)['occupation_name']:
        jobs = main_metro_scores[main_metro_scores['occupation_name'] == occ]['employment'].values[0]
        jobs_formatted = "{:,}".format(int(jobs))
        job_rate = main_metro_scores[main_metro_scores['occupation_name'] == occ]['employment_rate'].values[0]
        #format thousands and one decimal place
        job_rate_formatted = "{:,.1f}".format(float(job_rate))
        state_rate = main_metro_scores[main_metro_scores['occupation_name'] == occ]['state_emp_rate'].values[0]
        state_rate_formatted = "{:,.1f}".format(float(state_rate))
        state_compare = '🟢⬆️' if job_rate > state_rate else '🔴⬇️'
        ai_rating = main_metro_scores[main_metro_scores['occupation_name'] == occ]['study_rating'].values[0]
        ai_rating_formatted = "{:,.2f}".format(float(ai_rating))
        #format for thousands and convert to int for easier reading
        print('')
        print(f"{occ}")
        print(f"{jobs_formatted} workers")
        print(f"{job_rate_formatted} jobs per 10k workers vs. statewide rate of {state_rate_formatted} ({state_compare})")
        print(f"{ai_rating_formatted}")


Fast Food and Counter Workers
105,810 workers
321.6 jobs per 10k workers vs. statewide rate of 327.3 (🔴⬇️)
0.07

General and Operations Managers
97,320 workers
295.8 jobs per 10k workers vs. statewide rate of 306.5 (🔴⬇️)
0.38

Retail Salespersons
78,960 workers
240.0 jobs per 10k workers vs. statewide rate of 242.8 (🔴⬇️)
0.36

Stockers and Order Fillers
72,970 workers
221.8 jobs per 10k workers vs. statewide rate of 229.3 (🔴⬇️)
0.19

Registered Nurses
65,910 workers
200.4 jobs per 10k workers vs. statewide rate of 192.9 (🟢⬆️)
0.38

Customer Service Representatives
65,510 workers
199.1 jobs per 10k workers vs. statewide rate of 239.6 (🔴⬇️)
0.70


## Which occupations in my market have the highest AI exposure score?

I wouldn't expect there to be occupations unique to our main market here FYI. This is just establishing a baseline to get us used to the highest study ratings and how those present in our market.

In [13]:
emp_cols = ['area_name','occupation_name','employment',
            'employment_rate','state_emp_rate','study_rating']
for occ in main_metro_scores.sort_values('study_rating', ascending=False)[emp_cols].head(6)['occupation_name']:
        jobs = main_metro_scores[main_metro_scores['occupation_name'] == occ]['employment'].values[0]
        jobs_formatted = "{:,}".format(int(jobs)) if jobs > 0 else '0'
        job_rate = main_metro_scores[main_metro_scores['occupation_name'] == occ]['employment_rate'].values[0]
        #format thousands and one decimal place
        job_rate_formatted = "{:,.1f}".format(float(job_rate))
        state_rate = main_metro_scores[main_metro_scores['occupation_name'] == occ]['state_emp_rate'].values[0]
        state_rate_formatted = "{:,.1f}".format(float(state_rate))
        state_compare = '🟢⬆️' if job_rate > state_rate else '🔴⬇️'
        ai_rating = main_metro_scores[main_metro_scores['occupation_name'] == occ]['study_rating'].values[0]
        ai_rating_formatted = "{:,.2f}".format(float(ai_rating))
        #format for thousands and convert to int for easier reading
        print('')
        print(f"{occ}")
        print(f"{jobs_formatted} workers")
        print(f"{job_rate_formatted} jobs per 10k workers vs. statewide rate of {state_rate_formatted} ({state_compare})")
        print(f"{ai_rating_formatted}")


Survey Researchers
100 workers
0.3 jobs per 10k workers vs. statewide rate of 0.5 (🔴⬇️)
0.84

Interpreters and Translators
2,530 workers
7.7 jobs per 10k workers vs. statewide rate of 4.8 (🟢⬆️)
0.84

Writers and Authors
400 workers
1.2 jobs per 10k workers vs. statewide rate of 1.6 (🔴⬇️)
0.81

Public Relations Specialists
5,160 workers
15.7 jobs per 10k workers vs. statewide rate of 17.9 (🔴⬇️)
0.79

Legal Secretaries and Administrative Assistants
4,210 workers
12.8 jobs per 10k workers vs. statewide rate of 9.5 (🟢⬆️)
0.76

Executive Secretaries and Executive Administrative Assistants
7,390 workers
22.5 jobs per 10k workers vs. statewide rate of 24.5 (🔴⬇️)
0.74


## Which occupations have the worst combination of high AI exposure and large employment?

In [14]:
#filters = 'high ai, high employment'
filters = 'highish ai, high employment'
print_unique_occupations(
    your_main_metro=main_metro_scores, 
    your_similar_metros_list=similar_metros, 
    show_similar=True, 
    filters=filters,
    sort_col='employment',
    list_size=5
    )

There are 60 occupations in the main metro after applying filters. Here are the top 5 occupations based on employment:

Customer Service Representatives
65,510 workers
199.1 jobs per 10k workers vs. statewide rate of 239.6 (🔴⬇️)
0.70 AI exposure rating

First-Line Supervisors of Office and Administrative Support Workers
36,940 workers
112.3 jobs per 10k workers vs. statewide rate of 114.1 (🔴⬇️)
0.53 AI exposure rating

Secretaries and Administrative Assistants, Except Legal, Medical, and Executive
33,710 workers
102.5 jobs per 10k workers vs. statewide rate of 106.4 (🔴⬇️)
0.57 AI exposure rating

Sales Representatives of Services, Except Advertising, Insurance, Financial Services, and Travel
28,720 workers
87.3 jobs per 10k workers vs. statewide rate of 96.0 (🔴⬇️)
0.57 AI exposure rating

Accountants and Auditors
28,400 workers
86.3 jobs per 10k workers vs. statewide rate of 82.8 (🟢⬆️)
0.52 AI exposure rating
All occupations in the top 5 based on employment of the main metro are also i

,area_name,occupation_name,employment,study_rating,ai_exposure_category,employment_category
31556,"Chicago-Naperville-Elgin, IL-IN",Customer Service Representatives,75240.0,0.704545,Medium high,High
31531,"Chicago-Naperville-Elgin, IL-IN","Sales Representatives of Services, Except Advertising, Insurance, Financial Services, and Travel",55730.0,0.566667,Medium high,High
31582,"Chicago-Naperville-Elgin, IL-IN","Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",50190.0,0.567308,Medium high,High
31141,"Chicago-Naperville-Elgin, IL-IN",Accountants and Auditors,41360.0,0.520000,Medium high,High
31533,"Chicago-Naperville-Elgin, IL-IN","Sales Representatives, Wholesale and Manufacturing, Except Technical and Scientific Products",38490.0,0.706897,Medium high,High
39712,"Dallas-Fort Worth-Arlington, TX",Customer Service Representatives,96930.0,0.704545,Medium high,High
39699,"Dallas-Fort Worth-Arlington, TX",First-Line Supervisors of Office and Administrative Support Workers,45710.0,0.531250,Medium high,High
39689,"Dallas-Fort Worth-Arlington, TX","Sales Representatives of Services, Except Advertising, Insurance, Financial Services, and Travel",43560.0,0.566667,Medium high,High
39739,"Dallas-Fort Worth-Arlington, TX","Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",37550.0,0.567308,Medium high,High
39301,"Dallas-Fort Worth-Arlington, TX",Accountants and Auditors,37450.0,0.520000,Medium high,High


## What are the occupations in this metro that have a higher share of jobs than the state?

In [15]:
rename_cols = {'area_name':'Location', 'occupation_name':'Occupation','employment':'Jobs',
               'employment_category':'Employment category','study_rating':'AI exposure rating',
               'ai_exposure_category':'AI exposure category','employment_rate':'Jobs per 10k', 
               'state_emp_rate':'State jobs per 10k','state_metro_diff':'State-metro difference'}
more_jobs_than_state = main_metro_scores.loc[~main_metro_scores['occupation_code'].str.endswith('0000')].sort_values('state_metro_diff', ascending=False).head(10)
more_jobs_than_state.rename(columns=rename_cols, inplace=True)
display(more_jobs_than_state[rename_cols.values()])
more_jobs_than_state[rename_cols.values()].to_csv(f'../data/output/{market_state_abbr.lower()}/{market_metro_abbr}_top_10_occupations_with_larger_employment_in_metro_than_state.csv', index=False)

,Location,Occupation,Jobs,Employment category,AI exposure rating,AI exposure category,Jobs per 10k,State jobs per 10k,State-metro difference
390,"Houston-Pasadena-The Woodlands, TX","Janitors and Cleaners, Except Maids and Housekeeping Cleaners",48880.0,High,0.027778,Low,148.584074,131.360883,17.223191
504,"Houston-Pasadena-The Woodlands, TX",Construction Laborers,34080.0,High,0.025000,Low,103.595443,87.590504,16.004939
573,"Houston-Pasadena-The Woodlands, TX",Industrial Machinery Mechanics,18360.0,High,0.145161,Low,55.810221,40.472854,15.337367
382,"Houston-Pasadena-The Woodlands, TX",Waiters and Waitresses,51330.0,High,0.224490,Low,156.031516,141.857530,14.173986
642,"Houston-Pasadena-The Woodlands, TX",Chemical Equipment Operators and Tenders,9390.0,High,0.065217,Low,28.543463,14.838862,13.704601
366,"Houston-Pasadena-The Woodlands, TX",Security Guards,29250.0,High,0.250000,Low,88.913342,75.480628,13.432714
93,"Houston-Pasadena-The Woodlands, TX",Civil Engineers,12140.0,High,0.375000,Medium low,36.902837,24.781184,12.121653
212,"Houston-Pasadena-The Woodlands, TX","Elementary School Teachers, Except Special Education",30980.0,High,0.310811,Medium low,94.172148,82.622897,11.549252
616,"Houston-Pasadena-The Woodlands, TX","Welders, Cutters, Solderers, and Brazers",15850.0,High,0.039216,Low,48.180392,36.955020,11.225372
174,"Houston-Pasadena-The Woodlands, TX",Lawyers,16810.0,High,0.475000,Medium low,51.098574,40.209905,10.888669


## What share of total jobs in the metro have high exposure? Medium? Low?

I'm also curious how many occupations scored in the study fall into each of the ai exposure categories:

In [16]:
def get_ai_exposure_category(score):
    if score <= .25:
        return 'Low'
    elif score <= .5:
        return 'Medium low'
    elif score <= .75:
        return 'Medium high'
    elif score > .75:
        return 'High'
    else:
        return 'NA'

occ_scores = pd.read_csv('https://raw.githubusercontent.com/openai/GPTs-are-GPTs/refs/heads/main/data/occ_level.csv')
study_rating = 'human_rating_beta'
occ_scores = occ_scores[['O*NET-SOC Code', 'Title', study_rating]]
occ_scores['occupation_code'] = occ_scores['O*NET-SOC Code'].str.replace('-', '')#.str[:6]
occ_scores['occupation_code'] = occ_scores['occupation_code'].apply(lambda x: x if len(x) == 6 else x[:6])
occ_scores = occ_scores.rename(columns={study_rating: 'study_rating', 
                                        'O*NET-SOC Code': 'soc_code',
                                        'Title': 'occupation_name'})

all_occ_scores = occ_scores.copy()
all_occ_scores['ai_exposure_category'] = all_occ_scores['study_rating'].apply(get_ai_exposure_category)
# Apply the ordered category type
all_occ_scores['ai_exposure_category'] = all_occ_scores['ai_exposure_category'].astype(size_order)

all_occ_by_score = all_occ_scores.groupby('ai_exposure_category',dropna=False)['occupation_code'].count().reset_index()
all_occ_by_score['share'] = (all_occ_by_score['occupation_code'] / len(all_occ_scores))*100

print('Number of unique occupations in each AI exposure category:')
all_occ_by_score.sort_values('ai_exposure_category',ascending=False)

HTTPError: HTTP Error 429: Too Many Requests

In [ ]:
total_metro_emp = metro_scores.loc[(metro_scores['occupation_code'] == '000000')&(metro_scores['area_name'] == market_metro)]
metro_by_exposure = main_metro_scores.groupby('ai_exposure_category',dropna=False)['employment'].sum().reset_index()
metro_by_exposure['share'] = (metro_by_exposure['employment'] / total_metro_emp['employment'].values[0])*100

print(f'Share of total metro employment in each AI exposure category for {market_metro}:')
metro_by_exposure.sort_values('ai_exposure_category',ascending=False)

I should also probably look at these buckets statewide and nationwide, yeah?

In [ ]:
total_state_emp = state_scores.loc[(state_scores['occupation_code'] == '000000')&(state_scores['area_name'] == market_state)]
state_by_exposure = state_scores.loc[(state_scores['area_name'] == market_state)].groupby('ai_exposure_category',dropna=False)['employment'].sum().reset_index()
state_by_exposure['share'] = (state_by_exposure['employment'] / total_state_emp['employment'].values[0])*100

print(f'Share of total state employment in each AI exposure category for {market_state}:')
state_by_exposure.sort_values('ai_exposure_category',ascending=False)

In [ ]:
total_national_emp = national_scores.loc[(national_scores['occupation_code'] == '000000')]
national_by_exposure = national_scores.groupby('ai_exposure_category',dropna=False)['employment'].sum().reset_index()
national_by_exposure['share'] = (national_by_exposure['employment'] / total_national_emp['employment'].values[0])*100

print(f'Share of total national employment in each AI exposure category:')
national_by_exposure.sort_values('ai_exposure_category',ascending=False)

In [ ]:
#combine them all so we can compare better and turn into a quick chart
by_exposure = metro_by_exposure.merge(state_by_exposure, on='ai_exposure_category', how='left', suffixes=('_metro','_state'))
by_exposure = by_exposure.merge(national_by_exposure, on='ai_exposure_category', how='left')
by_exposure = by_exposure.rename(columns={'employment': 'emp_national', 'share': 'National','share_metro':'Metro','share_state':'State','share_national':'National'})
by_exposure = by_exposure[['ai_exposure_category','Metro','State','National']]
by_exposure = by_exposure.loc[by_exposure['ai_exposure_category'] != 'NA']


by_exposure.sort_values('ai_exposure_category',ascending=False).to_csv(f'../data/output/{market_state_abbr.lower()}/{market_metro_abbr}_employment_by_ai_exposure.csv', index=False)

## How does my metro's share of AI-exposed jobs compare to similarly-sized metros?

In [ ]:
total_metro_emp = metro_scores.loc[(metro_scores['occupation_code'] == '000000')&(metro_scores['area_name'] == market_metro)]
metro_by_exposure = main_metro_scores.groupby('ai_exposure_category',dropna=False)['employment'].sum().reset_index()
metro_by_exposure['share'] = (metro_by_exposure['employment'] / total_metro_emp['employment'].values[0])*100

print(f'Share of total metro employment in each AI exposure category for {market_metro}:')
metro_by_exposure.sort_values('ai_exposure_category',ascending=False)

In [ ]:
#checkout all similiarly-sized metros
for metro_area in similar_metros:
    print(metro_area)
    this_metro_scores = metro_scores[metro_scores['area_name'] == metro_area]
    total_metro_emp = this_metro_scores.loc[(this_metro_scores['occupation_code'] == '000000')]
    metro_by_exposure = this_metro_scores.groupby('ai_exposure_category',dropna=False)['employment'].sum().reset_index()
    metro_by_exposure['share'] = (metro_by_exposure['employment'] / total_metro_emp['employment'].values[0])*100

    print(f'Share of total metro employment in each AI exposure category for {metro_area}:')
    display(metro_by_exposure.sort_values('ai_exposure_category',ascending=False))
    # this_metro_rate = get_employment_rate(metro_scores.loc[metro_scores['area_name'] == metro_area], statewide=False)
    # display(this_metro_rate.groupby('ai_exposure_category',dropna=False)['employment_rate'].sum())

In [ ]:
metro_scores.head()

## Which top-level occupations (health care, service industry, etc) are most exposed in my metro?

In [ ]:
def get_exposure_job_share(group):
    return pd.Series({
        'share_high_exposure': (group[group['ai_exposure_category'] == 'High']['employment'].sum()/group['employment'].sum())*100,
        'share_medhigh_exposure': (group[group['ai_exposure_category'] == 'Medium high']['employment'].sum()/group['employment'].sum())*100,
        'share_medlow_exposure': (group[group['ai_exposure_category'] == 'Medium low']['employment'].sum()/group['employment'].sum())*100,
        'share_low_exposure': (group[group['ai_exposure_category'] == 'Low']['employment'].sum()/group['employment'].sum())*100
    })
by_top_occ_1 = main_metro_scores.groupby(['top_occ_code',
                                        'top_occ_name'],dropna=False).agg(total_jobs=('employment','sum'),
                                                                        occupations=('occupation_code','count'),
                                                                        avg_study_rating=('study_rating','mean'),
                                                                        median_study_rating=('study_rating','median')
                                                                        ).reset_index()
by_top_occ_2 = main_metro_scores.groupby(['top_occ_code',
                                          'top_occ_name'],dropna=False).apply(get_exposure_job_share).reset_index()
by_top_occ = by_top_occ_1.merge(by_top_occ_2, on=['top_occ_code','top_occ_name'], how='left')

In [ ]:
rename_cols = {'top_occ_code': 'Occupation group code', 'top_occ_name': 'Occupation group', 
               'occupations': 'Occupations in group','total_jobs': 'Jobs', 
               'avg_study_rating': 'Avg. study rating', 'share_high_exposure': '% high exposure', 
               'share_medhigh_exposure': '% medium-high exposure', 
               'share_medlow_exposure': '% medium-low exposure', 'share_low_exposure': '% low exposure'}
by_top_occ.rename(columns=rename_cols, inplace=True)
by_top_occ.sort_values('% medium-high exposure', ascending=False).to_csv(f'../data/output/{market_state_abbr.lower()}/{market_metro_abbr}_by_top_occ.csv', index=False)

In [ ]:
by_top_occ.sort_values('% medium-high exposure', ascending=False).head()